# Data Cleaning & Preprocessing — Online Retail II

**Author:** Aman Singh  
**Input:** `data/raw/online_retail_II.csv`  
**Target Output:** `data/interim/online_retail_cleaned.csv`

---

## Cleaning Objective & Guardrails
Based on the pre-cleaning audit in `notebooks/1.0-eda.ipynb`, raw records contain exact duplicates, missing `Customer ID`s (guest checkouts), missing/whitespace-padded `Description`s, zero/negative accounting entries, test SKUs, cancellations, and non-product fees.

Our goal is to produce a clean, analysis-ready interim dataset while keeping **total row drops strictly under 5%** of the raw dataset (`1,067,371` rows).

## Step 1: Environment Setup, Raw Data Loading & Baseline Tracker

In [1]:
import os
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 15)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

raw_path = 'data/raw/online_retail_II.csv' if os.path.exists('data/raw/online_retail_II.csv') else '../data/raw/online_retail_II.csv'
df_raw = pd.read_csv(raw_path)

# Keep raw DataFrame untouched; operate on a working copy
df = df_raw.copy()
baseline_rows = len(df_raw)

# Track retention after every cleaning step
retention_log = []
def record_step(step_name, current_df):
    curr_n = len(current_df)
    prev_n = retention_log[-1]['Remaining_Rows'] if retention_log else baseline_rows
    step_removed = prev_n - curr_n
    total_dropped = baseline_rows - curr_n
    retention_log.append({
        'Step': step_name,
        'Rows_Removed_Step': step_removed,
        'Remaining_Rows': curr_n,
        'Cumulative_Dropped_Pct': round((total_dropped / baseline_rows) * 100, 2),
        'Retained_Pct': round((curr_n / baseline_rows) * 100, 2)
    })
    return pd.DataFrame(retention_log)

print(f"Loaded from: {raw_path}")
print(f"Baseline Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
display(record_step("Step 1: Raw Baseline Loaded", df))

Loaded from: ../data/raw/online_retail_II.csv
Baseline Shape: 1,067,371 rows x 8 columns


,Step,Rows_Removed_Step,Remaining_Rows,Cumulative_Dropped_Pct,Retained_Pct
0,Step 1: Raw Baseline Loaded,0,1067371,0.000,100.000


**Step 1 Conclusion:**
- Raw dataset successfully loaded with **`1,067,371` rows** and **`8` columns**.
- We copy `df_raw` into `df` so the original in-memory snapshot remains immutable for side-by-side comparison.
- A `record_step()` audit tracker is initialized to monitor cumulative row drops after every step and verify we stay well under the **5%** drop limit.

## Step 2: Remove Exact Duplicate Transactions

In [2]:
dup_count = df.duplicated().sum()
print(f"Exact duplicate rows found: {dup_count:,} ({dup_count / baseline_rows * 100:.2f}% of raw)")

print("\n--- Example Duplicate Pair Before Removal (Invoice 489517) ---")
display(df[df.duplicated(keep=False) & (df['Invoice'] == '489517')].sort_values('StockCode').head(4))

# Drop exact duplicate rows across all 8 columns
df = df.drop_duplicates().reset_index(drop=True)

assert df.duplicated().sum() == 0, "Duplicates still remain!"
print(f"Deduplication verified. Remaining duplicates: {df.duplicated().sum()}")
display(record_step("Step 2: Drop Exact Duplicates", df))

Exact duplicate rows found: 34,335 (3.22% of raw)

--- Example Duplicate Pair Before Removal (Invoice 489517) ---


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.950,16329.000,United Kingdom
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.950,16329.000,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.750,16329.000,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.750,16329.000,United Kingdom


Deduplication verified. Remaining duplicates: 0


,Step,Rows_Removed_Step,Remaining_Rows,Cumulative_Dropped_Pct,Retained_Pct
0,Step 1: Raw Baseline Loaded,0,1067371,0.000,100.000
1,Step 2: Drop Exact Duplicates,34335,1033036,3.220,96.780


**Step 2 Conclusion:**
- **Result:** Removed **`34,335` exact duplicate rows (`3.22%`)**, leaving **`1,033,036` rows (`96.78%` retained)**.
- **Why full-row deduplication (`subset=None`):** A customer can legitimately purchase multiple different products under the same `Invoice` number, so we only drop rows where *every* column (`Invoice`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `Price`, `Customer ID`, `Country`) matches identically (e.g., POS double-scans or checkout network retries).
- **Impact:** Prevents artificial inflation of order line counts, total revenue, and RFM Frequency scores.